# 🤖 Pepê AI — Fine-tuning com QLoRA

Fine-tuning do **Llama 3 8B** com os dados do Pepê usando **Unsloth + QLoRA**.
Todos os arquivos são salvos no **Google Drive (pepe-ai)**.

> ⚠️ **Requisito:** Runtime → Alterar tipo de ambiente de execução → **GPU T4**

## ⚙️ Etapa 1 — Instalar dependências
> Após rodar esta célula, o runtime reinicia automaticamente. Continue da **Etapa 2** em diante.

In [ ]:
!nvidia-smi
!pip install -q protobuf==3.20.3
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "datasets>=2.16" "transformers>=4.40" accelerate bitsandbytes trl huggingface_hub
import os
print('✅ Instalação concluída. Reiniciando runtime...')
os.kill(os.getpid(), 9)

## 💾 Etapa 2 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# Google Drive / MyDrive / pepe-ai
D           = Path('/content/drive/MyDrive')
ROOT        = D / 'pepe-ai'
DATA        = ROOT / 'data'
CHECKPOINTS = ROOT / 'checkpoints'
LORA        = ROOT / 'lora'
GGUF        = ROOT / 'gguf'
LOGS        = ROOT / 'logs'
CACHE       = ROOT / 'cache'

for p in [ROOT, DATA, CHECKPOINTS, LORA, GGUF, LOGS, CACHE]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']            = str(CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(CACHE)

print('✅ Drive montado:')
for name, path in [('ROOT', ROOT), ('DATA', DATA), ('LORA', LORA), ('GGUF', GGUF), ('CACHE', CACHE)]:
    print(f'   {name} → {path}')

## 📦 Etapa 3 — Carregar dataset do GitHub

In [ ]:
import json, urllib.request
from datasets import Dataset

BASE_URL = 'https://raw.githubusercontent.com/Joao-Matheus-Amorim/pepe-ai/main/training/datasets/'

def load_jsonl_from_url(url):
    with urllib.request.urlopen(url) as r:
        return [json.loads(line) for line in r.read().decode().splitlines() if line.strip()]

train_data = load_jsonl_from_url(BASE_URL + 'pepe_train.jsonl')
val_data   = load_jsonl_from_url(BASE_URL + 'pepe_val.jsonl')

# Salva cópia local no Drive
with open(DATA / 'pepe_train.jsonl', 'w', encoding='utf-8') as f:
    for row in train_data: f.write(json.dumps(row, ensure_ascii=False) + '\n')
with open(DATA / 'pepe_val.jsonl', 'w', encoding='utf-8') as f:
    for row in val_data: f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'✅ Treino:    {len(train_data)} exemplos')
print(f'✅ Validação: {len(val_data)} exemplos')
print('\n--- Exemplo ---')
for msg in train_data[0]['messages']:
    role    = msg['role'].upper()
    content = msg.get('content') or '[tool_call]'
    print(f'[{role}] {content[:100]}')

## 🧠 Etapa 4 — Carregar Llama 3 8B + QLoRA

In [ ]:
import gc, torch
from unsloth import FastLanguageModel

# Limpeza de memória antes de carregar
gc.collect()
torch.cuda.empty_cache()
print(f'🧹 VRAM livre: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

MAX_SEQ_LENGTH = 1024  # Reduzido de 2048 para caber na T4 (15 GB VRAM)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/llama-3-8b-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
    cache_dir      = str(CACHE),
)

print(f'✅ Modelo carregado: {model.num_parameters()/1e6:.0f}M parâmetros')
print(f'   GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM livre: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA pronto: {trainable/1e6:.2f}M treináveis ({100*trainable/total:.2f}%)')

## 📝 Etapa 5 — Formatar dataset e treinar

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template='llama-3')

def format_messages(examples):
    texts = []
    for msgs in examples['messages']:
        clean = [m for m in msgs
                 if m.get('role') in ('system', 'user', 'assistant')
                 and m.get('content') is not None]
        texts.append(tokenizer.apply_chat_template(
            clean, tokenize=False, add_generation_prompt=False
        ))
    return {'text': texts}

train_hf = Dataset.from_list(train_data).map(format_messages, batched=True)
val_hf   = Dataset.from_list(val_data).map(format_messages, batched=True)
print(f'✅ Dataset formatado: {len(train_hf)} treino / {len(val_hf)} val')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_hf,
    eval_dataset       = val_hf,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 1,     # Reduzido de 2 → menos RAM de CPU
    packing            = True,  # Empacota sequências curtas → mais eficiente
    args = TrainingArguments(
        per_device_train_batch_size = 1,  # Reduzido de 2 → menos VRAM
        gradient_accumulation_steps = 8,  # Aumentado de 4 → batch efetivo igual (1×8=8)
        num_train_epochs            = 3,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        eval_strategy               = 'steps',
        eval_steps                  = 50,
        save_strategy               = 'steps',
        save_steps                  = 100,
        save_total_limit            = 3,
        output_dir                  = str(CHECKPOINTS),
        logging_dir                 = str(LOGS),
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        report_to                   = 'none',
    ),
)

print('🚀 Iniciando treinamento...')
stats = trainer.train()
print(f'\n✅ Concluído em {stats.metrics["train_runtime"]:.0f}s')
print(f'   Loss final: {stats.metrics["train_loss"]:.4f}')

## 💾 Etapa 6 — Salvar LoRA adapter no Drive

In [ ]:
model.save_pretrained(str(LORA))
tokenizer.save_pretrained(str(LORA))
print(f'✅ LoRA adapter salvo em: {LORA}')

import os
for f in os.listdir(LORA):
    size = os.path.getsize(LORA / f)
    print(f'   {f}: {size/1e6:.2f} MB')

## 🔄 Etapa 7 — Exportar GGUF Q4_K_M para o Drive

In [ ]:
print('🔄 Exportando GGUF Q4_K_M...')
print(f'   Destino: {GGUF}')

model.save_pretrained_gguf(
    str(GGUF),
    tokenizer,
    quantization_method = 'q4_k_m',
)

import os
print('\n✅ Arquivos gerados:')
for f in os.listdir(GGUF):
    size = os.path.getsize(GGUF / f) / 1e9
    print(f'   {f}: {size:.2f} GB')

## 🧪 Etapa 8 — Testar o modelo fine-tunado

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PEPE = (
    'Você é Pepê, um agente de IA pessoal inteligente, direto e eficiente. '
    'Você tem acesso a ferramentas de busca web, clima, visão de tela, execução de comandos, '
    'leitura de arquivos e memória persistente. '
    'Responda sempre em português do Brasil, de forma clara e concisa.'
)

def testar(pergunta):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PEPE},
        {'role': 'user',   'content': pergunta},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(
        input_ids      = inputs,
        max_new_tokens = 256,
        temperature    = 0.7,
        do_sample      = True,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split('assistant')[-1].strip()

perguntas = [
    'Quem é você?',
    'Quem criou você?',
    'Qual a stack do pepe-ai?',
    'O que você consegue fazer?',
    'Qual o próximo passo do projeto?',
]

for p in perguntas:
    print(f'\n[USER] {p}')
    print(f'[PEPÊ] {testar(p)}')

## 🚀 Etapa 9 — Integrar com Ollama no seu PC

Após o treino, o `.gguf` estará em:
```
Google Drive / MyDrive / pepe-ai / gguf / pepe-ft-unsloth.Q4_K_M.gguf
```

No **PowerShell do seu PC**:

```powershell
cd 'D:\pepe-ai\gguf'

@'
FROM ./pepe-ft-unsloth.Q4_K_M.gguf
SYSTEM "Você é Pepê, agente pessoal do João Matheus. Direto, brasileiro, competente."
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
'@ | Out-File -Encoding utf8 Modelfile

ollama create pepe-ft -f Modelfile
ollama run pepe-ft "Quem é você?"
```

Depois edite o `.env` do projeto:
```
PEPE_MODEL_PROVIDER=ollama
PEPE_MODEL_NAME=pepe-ft
```